# SAM3 バスケットボール追跡
**手順:**
1. 上メニュー「ランタイム」→「ランタイムのタイプを変更」→ **T4 GPU** を選択
2. セルを上から順に実行（▶ボタン or Shift+Enter）

In [ ]:
# セル1: GPU確認
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'なし（GPUランタイムを選択してください）')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')

In [ ]:
# セル2: ライブラリインストール
!pip install -q transformers>=5.5.0 opencv-python-headless

In [ ]:
# セル3: HuggingFace ログイン（SAM3アクセスに必要）
from huggingface_hub import login
# 下のトークンは使用後にhttps://huggingface.co/settings/tokensで削除してください
HF_TOKEN = "YOUR_HF_TOKEN"  # ← トークン
login(token=HF_TOKEN)
print('ログイン完了')

In [ ]:
# セル4: Google Drive をマウント
from google.colab import drive
drive.mount('/content/drive')
print('Drive マウント完了')
print('次のセルを実行する前に、DriveのMy Drive/basketball_analysis/に動画をアップロードしてください')

In [ ]:
# セル5: 動画パス確認
import os

# Google Drive内の動画パス（必要に応じて変更）
VIDEO_PATH = "/content/drive/MyDrive/basketball_analysis/game_EE1swQMsXJc_720p.mp4"

if os.path.exists(VIDEO_PATH):
    size_mb = os.path.getsize(VIDEO_PATH) / 1e6
    print(f'✓ 動画確認: {VIDEO_PATH} ({size_mb:.0f} MB)')
else:
    print(f'✗ 動画が見つかりません: {VIDEO_PATH}')
    print('Google DriveのMy Drive/basketball_analysis/フォルダに動画をアップロードしてください')

In [ ]:
# セル6: SAM3モデルロード
from transformers import Sam3Model, Sam3Processor
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'デバイス: {DEVICE}')

print('SAM3 読み込み中（初回は数分かかります）...')
processor = Sam3Processor.from_pretrained('facebook/sam3')
model = Sam3Model.from_pretrained('facebook/sam3', torch_dtype=torch.float16)
model = model.to(DEVICE)
model.eval()
print('SAM3 ロード完了')

In [ ]:
# セル7: SAM3 ボール追跡関数
import cv2
import numpy as np
from PIL import Image

BALL_PROMPT = "basketball"
FRAME_SKIP = 3  # 3フレームおきに推論（速度とのトレードオフ）

def infer_ball(frame_bgr):
    """1フレームでボール位置を推論 → (cx, cy) or None"""
    H, W = frame_bgr.shape[:2]
    img = Image.fromarray(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
    
    inputs = processor(
        images=img,
        text=BALL_PROMPT,
        return_tensors='pt',
    )
    inputs = {k: v.to(DEVICE).half() if v.dtype == torch.float32 else v.to(DEVICE)
              for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    try:
        results = processor.post_process_instance_segmentation(
            outputs, threshold=0.3, mask_threshold=0.5,
            target_sizes=[(H, W)]
        )[0]
    except Exception:
        return None

    if len(results['masks']) == 0:
        return None

    best_idx = int(results['scores'].argmax())
    mask = results['masks'][best_idx].cpu().numpy().astype(bool)
    if not mask.any():
        return None

    ys, xs = np.where(mask)
    area = len(xs)
    if not (20 < area < 8000):  # ボールらしいサイズのみ
        return None

    return (float(xs.mean()), float(ys.mean()))

print('関数定義完了')

In [ ]:
# セル8: 追跡実行（60秒分）
import json
from tqdm import tqdm

SEC = 300  # 何秒分処理するか

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
W   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
n_frames = int(SEC * fps)
print(f'{W}x{H} @ {fps:.1f}fps  処理: {n_frames}フレーム ({SEC}秒)')

ball_positions = {}
n_detected = 0

for fi in tqdm(range(n_frames), desc='SAM3追跡'):
    ret, frame = cap.read()
    if not ret:
        break

    if fi % FRAME_SKIP == 0:
        pos = infer_ball(frame)
        ball_positions[fi] = pos
        if pos:
            n_detected += 1
    else:
        ball_positions[fi] = ball_positions.get(fi - 1)

cap.release()

total = len(ball_positions)
print(f'\n追跡完了: {n_detected}/{total} ({100*n_detected/max(total,1):.1f}%)')

In [ ]:
# セル9: 結果をJSONで保存してダウンロード
from google.colab import files

out_path = '/content/ball_positions_sam3_60s.json'
serializable = {str(k): list(v) if v else None for k, v in ball_positions.items()}
with open(out_path, 'w') as f:
    json.dump(serializable, f)

print(f'保存完了: {out_path}')
files.download(out_path)  # ← 自動でダウンロードされます
print('ダウンロード開始！')

## ダウンロードしたら

ローカルの `basketball_analysis/` フォルダに `ball_positions_sam3_60s.json` を置いて：

```bash
python analyze_roboflow.py --sec 60 --sam3-pos ball_positions_sam3_60s.json
```

これでSAM3のボール位置でシュートチャートが生成されます！